Exercise 1 — Controlled operations

- Construct the CNOT matrix.
- Verify its unitarity.
- Test all four computational basis inputs.
- Construct the CZ matrix and test its action on all four basis states.

In [ ]:
import numpy as np

# Basis states
zero = np.array([1, 0], dtype=complex)
one = np.array([0, 1], dtype=complex)

# Tensor products of basis states
zero_zero = np.kron(zero, zero)
zero_one = np.kron(zero, one)
one_zero = np.kron(one, zero)
one_one = np.kron(one, one)

CNOT = np.array([
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 0, 1],
    [0, 0, 1, 0]
], dtype=complex)

print("CNOT unitarity:", np.allclose(CNOT @ CNOT.conj().T, np.eye(4)))

print("CNOT on |00>:", CNOT @ zero_zero)
print("CNOT on |01>:", CNOT @ zero_one)
print("CNOT on |10>:", CNOT @ one_zero)
print("CNOT on |11>:", CNOT @ one_one)

CZ = np.array([
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 1, 0],
    [0, 0, 0, -1]
], dtype=complex)

print("\nCZ unitarity:", np.allclose(CZ @ CZ.conj().T, np.eye(4)))

print("CZ on |00>:", CZ @ zero_zero)
print("CZ on |01>:", CZ @ zero_one)
print("CZ on |10>:", CZ @ one_zero)
print("CZ on |11>:", CZ @ one_one)


CNOT unitarity: True
CNOT on |00>: [1.+0.j 0.+0.j 0.+0.j 0.+0.j]
CNOT on |01>: [0.+0.j 1.+0.j 0.+0.j 0.+0.j]
CNOT on |10>: [0.+0.j 0.+0.j 0.+0.j 1.+0.j]
CNOT on |11>: [0.+0.j 0.+0.j 1.+0.j 0.+0.j]
CZ unitarity: True
CZ on |00>: [1.+0.j 0.+0.j 0.+0.j 0.+0.j]
CZ on |01>: [0.+0.j 1.+0.j 0.+0.j 0.+0.j]
CZ on |10>: [0.+0.j 0.+0.j 1.+0.j 0.+0.j]
CZ on |11>: [ 0.+0.j  0.+0.j  0.+0.j -1.+0.j]


Exercise 2 — SWAP gate

- Construct the SWAP matrix.
- Verify its unitarity.
- Test its action on all four computational basis states.
- Verify that applying SWAP twice restores the original state.

In [ ]:
SWAP = np.array([
    [1, 0, 0, 0],
    [0, 0, 1, 0],
    [0, 1, 0, 0],
    [0, 0, 0, 1]
], dtype=complex)

print("\nSWAP unitarity:", np.allclose(SWAP @ SWAP.conj().T, np.eye(4)))

print("SWAP on |00>:", SWAP @ zero_zero)
print("SWAP on |01>:", SWAP @ zero_one)
print("SWAP on |10>:", SWAP @ one_zero)
print("SWAP on |11>:", SWAP @ one_one)

print("Double SWAP restores original state:", np.allclose(SWAP @ SWAP @ zero_zero, zero_zero))


SWAP unitarity: True
SWAP on |00>: [1.+0.j 0.+0.j 0.+0.j 0.+0.j]
SWAP on |01>: [0.+0.j 0.+0.j 1.+0.j 0.+0.j]
SWAP on |10>: [0.+0.j 1.+0.j 0.+0.j 0.+0.j]
SWAP on |11>: [0.+0.j 0.+0.j 0.+0.j 1.+0.j]
Double SWAP restore original state: True


Exercise 3 — Entanglement through controlled gates

- Start with \(|00\rangle\).
- Apply \(H\otimes I\).
- Apply CNOT.
- Verify that the resulting state is \(|\Phi^+\rangle\).
- Repeat using \(H\otimes H\), followed by CZ.
- Verify the resulting state and its entanglement.

In [6]:
def density_matrix(state):
    return np.outer(state, state.conj())


def measurement_probabilities(state):
    return np.abs(state) ** 2


def purity(rho):
    return np.real(np.trace(rho @ rho))


def check_density_matrix(rho):
    hermitian = np.allclose(rho, rho.conj().T)
    trace_one = np.isclose(np.trace(rho), 1)
    eigenvalues = np.linalg.eigvalsh(rho)
    positive = np.all(eigenvalues >= -1e-10)

    return hermitian and trace_one and positive


def partial_trace(rho, keep):
    rho_4d = rho.reshape(2, 2, 2, 2)

    if keep == 0:
        return np.einsum('abcb->ac', rho_4d)
    elif keep == 1:
        return np.einsum('abad->bd', rho_4d)
    else:
        raise ValueError("keep must be 0 or 1")


def analyze_pure_two_qubit_state(state):
    rho = density_matrix(state)

    rho_A = partial_trace(rho, keep=0)
    rho_B = partial_trace(rho, keep=1)

    return {
        "joint_density_matrix": rho,
        "joint_purity": purity(rho),
        "reduced_A": rho_A,
        "reduced_B": rho_B,
        "purity_A": purity(rho_A),
        "purity_B": purity(rho_B),
        "entangled": (
            purity(rho_A) < 1 - 1e-10
        )
    }

In [ ]:
# Identity
I = np.eye(2, dtype=complex)

H = (1 / np.sqrt(2)) * np.array([
    [1, 1],
    [1, -1]
], dtype=complex)

H_first = np.kron(H, I)
H_first_00 = H_first @ zero_zero
result_state = CNOT @ H_first_00

bell_state = (1 / np.sqrt(2)) * (zero_zero + one_one)
print("Result state is bell state:", np.allclose(result_state, bell_state))

double_H = np.kron(H, H)
double_H_00 = double_H @ zero_zero
result_state_2 = CZ @ double_H_00

print("Result state is bell state:", np.allclose(result_state_2, bell_state))
print("Entangled state:", analyze_pure_two_qubit_state(result_state_2)['entangled'])

Result state is bell state: True
Result state is bell state: False
Analysis of the resulting state: True


Exercise 4 — Circuit construction

Implement a reusable function that accepts an initial two-qubit state and a sequence of gates, applies the gates in the correct order, and returns the final state.

Test it with at least three different gate sequences.

In [10]:
def circuit_sequence(initial_state, gates):
    state = initial_state
    for gate in gates:
        state = gate @ state
    return state

CZ_HH_00 = circuit_sequence(zero_zero, [double_H, CZ])
print("Double H followed by CZ on |00> gives:", CZ_HH_00)

H_CZ_CNOT_01 = circuit_sequence(zero_one, [H_first, CZ, CNOT])
print("H followed by CZ and CNOT on |01> gives:", H_CZ_CNOT_01)

CNOT_CZ_H_10 = circuit_sequence(zero_one, [CNOT, CZ, H_first])
print("CNOT followed by CZ and H on |01> gives:", CNOT_CZ_H_10)

Double H followed by CZ on |00> gives: [ 0.5+0.j  0.5+0.j  0.5+0.j -0.5+0.j]
H followed by CZ and CNOT on |01> gives: [ 0.        +0.j  0.70710678+0.j -0.70710678+0.j  0.        +0.j]
CNOT followed by CZ and H on |01> gives: [0.        +0.j 0.70710678+0.j 0.        +0.j 0.70710678+0.j]


Exercise 7 — Challenge

Construct the four Bell states using combinations of H, X, Z and CNOT.

Verify their normalization and orthogonality.

In [12]:
def is_normalized(state):
    probabilities = measurement_probabilities(state)
    return np.isclose(np.sum(probabilities), 1.0)

# Single-qubit gates
X = np.array([
    [0, 1],
    [1, 0]
], dtype=complex)

Z = np.array([
    [1, 0],
    [0, -1]
], dtype=complex)

phi_plus = CNOT @ H_first_00

phi_minus = np.kron(Z, I) @ phi_plus

psi_plus = np.kron(I, X) @ phi_plus

psi_minus = np.kron(Z, I) @ psi_plus

print("Phi+:", phi_plus)
print("Measurement probabilities for Phi+:", measurement_probabilities(phi_plus))
print("Is Phi+ normalized?", is_normalized(phi_plus))

print("\nPhi-:", phi_minus)
print("Measurement probabilities for Phi-:", measurement_probabilities(phi_minus))
print("Is Phi- normalized?", is_normalized(phi_minus))

print("\nPsi+:", psi_plus)
print("Measurement probabilities for Psi+:", measurement_probabilities(psi_plus))
print("Is Psi+ normalized?", is_normalized(psi_plus))

print("\nPsi-:", psi_minus)
print("Measurement probabilities for Psi-:", measurement_probabilities(psi_minus))
print("Is Psi- normalized?", is_normalized(psi_minus))

Phi+: [0.70710678+0.j 0.        +0.j 0.        +0.j 0.70710678+0.j]
Measurement probabilities for Phi+: [0.5 0.  0.  0.5]
Is Phi+ normalized? True

Phi-: [ 0.70710678+0.j  0.        +0.j  0.        +0.j -0.70710678+0.j]
Measurement probabilities for Phi-: [0.5 0.  0.  0.5]
Is Phi- normalized? True

Psi+: [0.        +0.j 0.70710678+0.j 0.70710678+0.j 0.        +0.j]
Measurement probabilities for Psi+: [0.  0.5 0.5 0. ]
Is Psi+ normalized? True

Psi-: [ 0.        +0.j  0.70710678+0.j -0.70710678+0.j  0.        +0.j]
Measurement probabilities for Psi-: [0.  0.5 0.5 0. ]
Is Psi- normalized? True
